In [18]:
import os
import sys

#  Set the working directory to the folder containing the top-level ultralytics package
os.chdir("/workspace")  # change to your workspace root where ultralytics folder exists
print("Current working directory:", os.getcwd())


Current working directory: /workspace


In [2]:
!git clone https://github.com/danielsyahputra/yolo-distiller.git
%cd yolo-distiller


Cloning into 'yolo-distiller'...
remote: Enumerating objects: 760, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 760 (delta 0), reused 0 (delta 0), pack-reused 757 (from 3)
Receiving objects: 100% (760/760), 1.68 MiB | 92.00 KiB/s, done.
Resolving deltas: 100% (185/185), done.
/workspace/yolo-distiller


/usr/local/lib/python3.10/dist-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [19]:
import os
print("Current working directory:", os.getcwd())


Current working directory: /workspace


In [20]:
#  Add the top-level ultralytics folder to Python path
os.chdir("/workspace/yolo-distiller")
print("Current working directory:", os.getcwd())

Current working directory: /workspace/yolo-distiller


In [21]:
from ultralytics import YOLO

import ultralytics
print(ultralytics.__file__)


/workspace/yolo-distiller/ultralytics/__init__.py


In [22]:
from ultralytics.nn import tasks
from ultralytics.models.yolo.model import YOLO

model = YOLO("bestv8neiou.pt")
model.info()


Model summary: 225 layers, 3,012,213 parameters, 0 gradients


(225, 3012213, 0, 0.0)

In [23]:
from ultralytics.nn import tasks
from ultralytics.models.yolo.model import YOLO

model = YOLO("bestv8leiou.pt")
model.info()


Model summary: 365 layers, 43,635,237 parameters, 0 gradients


(365, 43635237, 0, 0.0)

In [24]:
import os
print(os.getcwd())


/workspace/yolo-distiller


In [25]:
from ultralytics import YOLO
import time
import torch

# === Device setup ===
device = "cuda:0" if torch.cuda.is_available() else "cpu"

# === Load teacher and student models ===
teacher_model = YOLO("bestv8leiou.pt")        # Teacher: already trained v8l with EIoU
student_model = YOLO("bestv8neiou.pt")        # Student: already trained v8n with EIoU

# === Training parameters ===
dataset_yaml = "/workspace/datasets/KITTI/kitti.yml"
run_name = "yolov8nbase_noeiouloss_KD_hyperpat20"  # Run name for saving logs and weights
epochs = 1
imgsz = 1280
batch_size = 32
workers = 2
patience = 20
save_interval = 50
amp = True

# === Start KD training ===
print("\n=== STARTING KNOWLEDGE DISTILLATION TRAINING ===")
start_time = time.time()

student_model.train(
    data=dataset_yaml,
    teacher=teacher_model,
    distillation_loss="cwd",   # Use CWD for feature map distillation
    epochs=epochs,
    batch=batch_size,
    imgsz=imgsz,
    workers=workers,
    device=device,
    amp=amp,
    patience=patience,
    save_period=save_interval,
    optimizer="AdamW",
    lr0=0.001,
    weight_decay=1e-4,
    cos_lr=True,
    lrf=0.01,
    augment=True,
    name=run_name
)

total_time = time.time() - start_time
print(f"\nKD Training completed in {total_time/3600:.2f} hours")



=== STARTING KNOWLEDGE DISTILLATION TRAINING ===
New https://pypi.org/project/ultralytics/8.3.203 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.24 🚀 Python-3.10.12 torch-2.4.0a0+f70bd71a48.nv24.06 CUDA:0 (NVIDIA A100-SXM4-80GB, 81051MiB)
engine/trainer: task=detect, mode=train, model=bestv8neiou.pt, data=/workspace/datasets/KITTI/kitti.yml, epochs=1, time=None, patience=20, batch=32, imgsz=1280, save=True, save_period=50, cache=False, device=cuda:0, workers=2, project=None, name=yolov8nbase_noeiouloss_KD_hyperpat20, exist_ok=False, pretrained=True, optimizer=AdamW, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=True, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False

train: Scanning /workspace/datasets/KITTI/labels/train.cache... 5984 images, 0 backgrounds, 0 corrupt: 100%|██████████| 5984/5984 [00:00<?, ?it/s]
val: Scanning /workspace/datasets/KITTI/labels/val.cache... 1497 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1497/1497 [00:00<?, ?it/s]


Plotting labels to /workspace/yolo-distiller/runs/detect/yolov8nbase_noeiouloss_KD_hyperpat20/labels.jpg... 
optimizer: AdamW(lr=0.001, momentum=0.937) with parameter groups 154 weight(decay=0.0), 168 weight(decay=0.0001), 166 bias(decay=0.0)
TensorBoard: model graph visualization added ✅
Image sizes 1280 train, 1280 val
Using 2 dataloader workers
Logging results to /workspace/yolo-distiller/runs/detect/yolov8nbase_noeiouloss_KD_hyperpat20
Starting training for 1 epochs...

WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.780738830566406. Dividing input by 255.
0: 640x640 (no detections), 8.9ms
Speed: 0.0ms preprocess, 8.9ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 640)

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


  0%|          | 0/187 [00:00<?, ?it/s]


0: 1280x1280 20 Cars, 11.4ms
1: 1280x1280 1 Car, 1 Van, 1 Person_sitting, 11.4ms
2: 1280x1280 9 Cars, 11.4ms
3: 1280x1280 8 Cars, 1 Van, 1 Person_sitting, 11.4ms
4: 1280x1280 9 Cars, 1 Van, 1 Pedestrian, 1 Person_sitting, 1 Cyclist, 11.4ms
5: 1280x1280 8 Cars, 1 Van, 1 Pedestrian, 11.4ms
6: 1280x1280 3 Cars, 1 Van, 1 Pedestrian, 2 Person_sittings, 11.4ms
7: 1280x1280 3 Cars, 1 Van, 1 Pedestrian, 2 Person_sittings, 1 Cyclist, 11.4ms
8: 1280x1280 8 Cars, 1 Van, 3 Pedestrians, 2 Person_sittings, 11.4ms
9: 1280x1280 10 Cars, 1 Van, 1 Person_sitting, 11.4ms
10: 1280x1280 2 Cars, 11.4ms
11: 1280x1280 3 Pedestrians, 11.4ms
12: 1280x1280 7 Cars, 11.4ms
13: 1280x1280 8 Cars, 1 Pedestrian, 1 Cyclist, 11.4ms
14: 1280x1280 9 Cars, 1 Pedestrian, 11.4ms
15: 1280x1280 4 Cars, 1 Person_sitting, 11.4ms
16: 1280x1280 16 Cars, 2 Vans, 1 Pedestrian, 11.4ms
17: 1280x1280 4 Cars, 1 Pedestrian, 1 Cyclist, 11.4ms
18: 1280x1280 3 Cars, 2 Trucks, 1 Tram, 11.4ms
19: 1280x1280 10 Cars, 1 Van, 1 Truck, 11.4ms
20:

...


0: 1280x1280 10 Cars, 1 Truck, 3 Pedestrians, 11.3ms
1: 1280x1280 3 Cars, 2 Vans, 7 Pedestrians, 6 Cyclists, 11.3ms
2: 1280x1280 7 Cars, 11.3ms
3: 1280x1280 3 Cars, 2 Cyclists, 11.3ms
4: 1280x1280 15 Cars, 1 Van, 11.3ms
5: 1280x1280 7 Cars, 1 Van, 1 Truck, 1 Cyclist, 11.3ms
6: 1280x1280 7 Cars, 11.3ms
7: 1280x1280 3 Cars, 11.3ms
8: 1280x1280 6 Cars, 11.3ms
9: 1280x1280 14 Cars, 1 Van, 1 Truck, 1 Cyclist, 11.3ms
10: 1280x1280 3 Cars, 2 Vans, 11.3ms
11: 1280x1280 14 Cars, 11.3ms
12: 1280x1280 6 Cars, 1 Van, 11.3ms
13: 1280x1280 6 Cars, 1 Van, 4 Pedestrians, 1 Cyclist, 11.3ms
14: 1280x1280 20 Cars, 4 Vans, 1 Pedestrian, 11.3ms
15: 1280x1280 4 Cars, 1 Van, 1 Pedestrian, 11.3ms
16: 1280x1280 2 Cars, 11.3ms
17: 1280x1280 2 Cars, 1 Van, 13 Pedestrians, 1 Person_sitting, 2 Cyclists, 11.3ms
18: 1280x1280 9 Cars, 4 Vans, 1 Truck, 8 Pedestrians, 2 Person_sittings, 2 Cyclists, 11.3ms
19: 1280x1280 3 Pedestrians, 1 Cyclist, 11.3ms
20: 1280x1280 6 Cars, 2 Vans, 11.3ms
21: 1280x1280 12 Cars, 1 Truck

        1/1        28G     0.6292     0.3996     0.8932        360       1280: 100%|██████████| 187/187 [02:37<00:00,  1.19it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 24/24 [00:06<00:00,  3.47it/s]


                   all       1497       7772      0.859      0.853      0.899      0.687

1 epochs completed in 0.047 hours.
Optimizer stripped from /workspace/yolo-distiller/runs/detect/yolov8nbase_noeiouloss_KD_hyperpat20/weights/last.pt, 6.3MB
Optimizer stripped from /workspace/yolo-distiller/runs/detect/yolov8nbase_noeiouloss_KD_hyperpat20/weights/best.pt, 6.3MB

Validating /workspace/yolo-distiller/runs/detect/yolov8nbase_noeiouloss_KD_hyperpat20/weights/best.pt...
Ultralytics 8.3.24 🚀 Python-3.10.12 torch-2.4.0a0+f70bd71a48.nv24.06 CUDA:0 (NVIDIA A100-SXM4-80GB, 81051MiB)
Model summary (fused): 168 layers, 3,007,013 parameters, 0 gradients


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 24/24 [00:08<00:00,  2.97it/s]


                   all       1497       7772      0.885      0.848       0.91      0.688
                   Car       1333       5680      0.893       0.96      0.976      0.816
                   Van        425        563      0.866      0.932      0.958      0.785
                 Truck        190        198      0.931      0.904      0.964      0.823
            Pedestrian        357        896      0.907       0.74      0.867      0.526
        Person_sitting         13         30      0.769        0.7      0.746      0.438
               Cyclist        222        306      0.916      0.794      0.903      0.665
                  Tram         76         99      0.909       0.91      0.957      0.767
Speed: 0.4ms preprocess, 1.4ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to /workspace/yolo-distiller/runs/detect/yolov8nbase_noeiouloss_KD_hyperpat20

KD Training completed in 0.07 hours


In [26]:
from ultralytics import YOLO
import time
import torch

# === Device setup ===
device = "cuda:0" if torch.cuda.is_available() else "cpu"

# === Load teacher and student models ===
teacher_model = YOLO("bestv8leiou.pt")        # Teacher: already trained v8l with EIoU
student_model = YOLO("bestv8neiou.pt")        # Student: already trained v8n with EIoU

# === Training parameters ===
dataset_yaml = "/workspace/datasets/KITTI/kitti.yml"
run_name = "yolov8nbase_noeiouloss_KD_hyperpat20"  # Run name for saving logs and weights
epochs = 250
imgsz = 1280
batch_size = 32
workers = 2
patience = 20
save_interval = 50
amp = True

# === Start KD training ===
print("\n=== STARTING KNOWLEDGE DISTILLATION TRAINING ===")
start_time = time.time()

student_model.train(
    data=dataset_yaml,
    teacher=teacher_model,
    distillation_loss="cwd",   # Use CWD for feature map distillation
    epochs=epochs,
    batch=batch_size,
    imgsz=imgsz,
    workers=workers,
    device=device,
    amp=amp,
    patience=patience,
    save_period=save_interval,
    optimizer="AdamW",
    lr0=0.001,
    weight_decay=1e-4,
    cos_lr=True,
    lrf=0.01,
    augment=True,
    name=run_name
)

total_time = time.time() - start_time
print(f"\nKD Training completed in {total_time/3600:.2f} hours")



=== STARTING KNOWLEDGE DISTILLATION TRAINING ===
New https://pypi.org/project/ultralytics/8.3.203 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.24 🚀 Python-3.10.12 torch-2.4.0a0+f70bd71a48.nv24.06 CUDA:0 (NVIDIA A100-SXM4-80GB, 81051MiB)
engine/trainer: task=detect, mode=train, model=bestv8neiou.pt, data=/workspace/datasets/KITTI/kitti.yml, epochs=250, time=None, patience=20, batch=32, imgsz=1280, save=True, save_period=50, cache=False, device=cuda:0, workers=2, project=None, name=yolov8nbase_noeiouloss_KD_hyperpat20, exist_ok=False, pretrained=True, optimizer=AdamW, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=True, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=Fal

train: Scanning /workspace/datasets/KITTI/labels/train.cache... 5984 images, 0 backgrounds, 0 corrupt: 100%|██████████| 5984/5984 [00:00<?, ?it/s]
val: Scanning /workspace/datasets/KITTI/labels/val.cache... 1497 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1497/1497 [00:00<?, ?it/s]


Plotting labels to /workspace/yolo-distiller/runs/detect/yolov8nbase_noeiouloss_KD_hyperpat20/labels.jpg... 
optimizer: AdamW(lr=0.001, momentum=0.937) with parameter groups 154 weight(decay=0.0), 168 weight(decay=0.0001), 166 bias(decay=0.0)
TensorBoard: model graph visualization added ✅
Image sizes 1280 train, 1280 val
Using 2 dataloader workers
Logging results to /workspace/yolo-distiller/runs/detect/yolov8nbase_noeiouloss_KD_hyperpat20
Starting training for 250 epochs...

WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.780738830566406. Dividing input by 255.
0: 640x640 (no detections), 9.0ms
Speed: 0.0ms preprocess, 9.0ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 640)

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


  0%|          | 0/187 [00:00<?, ?it/s]


0: 1280x1280 20 Cars, 11.3ms
1: 1280x1280 1 Car, 1 Van, 1 Person_sitting, 11.3ms
2: 1280x1280 9 Cars, 11.3ms
3: 1280x1280 8 Cars, 1 Van, 1 Person_sitting, 11.3ms
4: 1280x1280 9 Cars, 1 Van, 1 Pedestrian, 1 Person_sitting, 1 Cyclist, 11.3ms
5: 1280x1280 8 Cars, 1 Van, 1 Pedestrian, 11.3ms
6: 1280x1280 3 Cars, 1 Van, 1 Pedestrian, 2 Person_sittings, 11.3ms
7: 1280x1280 3 Cars, 1 Van, 1 Pedestrian, 2 Person_sittings, 1 Cyclist, 11.3ms
8: 1280x1280 8 Cars, 1 Van, 3 Pedestrians, 2 Person_sittings, 11.3ms
9: 1280x1280 10 Cars, 1 Van, 1 Person_sitting, 11.3ms
10: 1280x1280 2 Cars, 11.3ms
11: 1280x1280 3 Pedestrians, 11.3ms
12: 1280x1280 7 Cars, 11.3ms
13: 1280x1280 8 Cars, 1 Pedestrian, 1 Cyclist, 11.3ms
14: 1280x1280 9 Cars, 1 Pedestrian, 11.3ms
15: 1280x1280 4 Cars, 1 Person_sitting, 11.3ms
16: 1280x1280 16 Cars, 2 Vans, 1 Pedestrian, 11.3ms
17: 1280x1280 4 Cars, 1 Pedestrian, 1 Cyclist, 11.3ms
18: 1280x1280 3 Cars, 2 Trucks, 1 Tram, 11.3ms
19: 1280x1280 10 Cars, 1 Van, 1 Truck, 11.3ms
20:

...

     22/250      28.7G     0.6368     0.3992     0.8972        294       1280: 100%|██████████| 187/187 [02:32<00:00,  1.23it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 24/24 [00:04<00:00,  5.28it/s]

                   all       1497       7772      0.881      0.884      0.921      0.709
EarlyStopping: Training stopped early as no improvement observed in last 20 epochs. Best results observed at epoch 2, best model saved as best.pt.
To update EarlyStopping(patience=20) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



22 epochs completed in 0.964 hours.
Optimizer stripped from /workspace/yolo-distiller/runs/detect/yolov8nbase_noeiouloss_KD_hyperpat20/weights/last.pt, 6.3MB
Optimizer stripped from /workspace/yolo-distiller/runs/detect/yolov8nbase_noeiouloss_KD_hyperpat20/weights/best.pt, 6.3MB

Validating /workspace/yolo-distiller/runs/detect/yolov8nbase_noeiouloss_KD_hyperpat20/weights/best.pt...
Ultralytics 8.3.24 🚀 Python-3.10.12 torch-2.4.0a0+f70bd71a48.nv24.06 CUDA:0 (NVIDIA A100-SXM4-80GB, 81051MiB)
Model summary (fused): 168 layers, 3,007,013 parameters, 0 gradients


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 24/24 [00:07<00:00,  3.15it/s]


                   all       1497       7772      0.894      0.873      0.923      0.715
                   Car       1333       5680      0.947      0.952      0.981      0.834
                   Van        425        563      0.938       0.94      0.969      0.813
                 Truck        190        198      0.937      0.939       0.97      0.828
            Pedestrian        357        896       0.84      0.784      0.869      0.535
        Person_sitting         13         30       0.78      0.707      0.796      0.518
               Cyclist        222        306      0.913      0.861      0.932      0.689
                  Tram         76         99      0.902      0.929      0.942       0.79
Speed: 0.1ms preprocess, 1.4ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to /workspace/yolo-distiller/runs/detect/yolov8nbase_noeiouloss_KD_hyperpat20

KD Training completed in 0.97 hours


In [28]:
import sys
from pathlib import Path
import torch
import time
from ultralytics import YOLO  # Use YOLOv8's interface

weights = "runs/detect/yolov8nbase_noeiouloss_KD_hyperpat20/weights/best.pt"
dataset_yaml = "/workspace/datasets/KITTI/kitti.yml"
results_dir = "/workspace/yolov8nbase_noeiouloss_KD_hyperpat20_distill"

# ------------------- Device -------------------
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# ------------------- Load model -------------------
model = YOLO(weights)  # Load the model using YOLOv8
params = sum(p.numel() for p in model.model.parameters())
params_mb = params * 4 / (1024**2)  # float32 -> 4 bytes
model_size_mb = Path(weights).stat().st_size / 1024**2  # file size in MB

print(f"Model size (file): {model_size_mb:.2f} MB")
print(f"Model parameters: {params_mb:.2f} MB")

# ------------------- FPS Measurement -------------------
dummy_input = torch.randn(1, 3, 1280, 1280).to(device) / 255.0  # Normalize the dummy input
# Warm-up
for _ in range(5):
    _ = model(dummy_input)  # Use the model directly for inference

n_runs = 50
start_time = time.time()
for _ in range(n_runs):
    _ = model(dummy_input)  # Use the model directly for inference
end_time = time.time()

fps = n_runs / (end_time - start_time)
print(f"Inference FPS (1280x1280): {fps:.2f}")

# ------------------- Run Validation -------------------
print("\nRunning YOLOv8 validation...")
start_val_time = time.time()
model.val(
    data=dataset_yaml,  # Validation dataset
    imgsz=1280,          # Image size
    batch=64,            # Corrected batch size argument
    device=device,       # Device
    project=results_dir, # Results directory
    name="eval_metrics", # Name for saved results
    save_json=True,      # Save results as JSON
    exist_ok=True,       # Overwrite if results directory exists
    verbose=True         # Show verbose output
)
end_val_time = time.time()

val_time = end_val_time - start_val_time
print(f"Validation time: {val_time:.2f} seconds")
print(f"Validation results saved to {results_dir}")


Model size (file): 5.99 MB
Model parameters: 11.49 MB

0: 1280x1280 (no detections), 4.9ms
Speed: 0.0ms preprocess, 4.9ms inference, 6.2ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 (no detections), 5.1ms
Speed: 0.0ms preprocess, 5.1ms inference, 1.5ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 (no detections), 5.1ms
Speed: 0.0ms preprocess, 5.1ms inference, 1.3ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 (no detections), 5.0ms
Speed: 0.0ms preprocess, 5.0ms inference, 1.3ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 (no detections), 5.1ms
Speed: 0.0ms preprocess, 5.1ms inference, 1.3ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 (no detections), 5.0ms
Speed: 0.0ms preprocess, 5.0ms inference, 1.2ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 (no detections), 5.1ms
Speed: 0.0ms preprocess, 5.1ms inference, 1.2ms postprocess per image at shape (1, 3, 1280, 1280)



val: Scanning /workspace/datasets/KITTI/labels/val.cache... 1497 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1497/1497 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 24/24 [00:08<00:00,  2.91it/s]


                   all       1497       7772      0.907      0.889      0.931      0.723
                   Car       1333       5680      0.943      0.958      0.982      0.847
                   Van        425        563      0.951      0.938       0.97      0.827
                 Truck        190        198      0.939       0.93       0.97      0.842
            Pedestrian        357        896      0.874       0.82      0.886      0.539
        Person_sitting         13         30      0.821      0.767      0.832      0.503
               Cyclist        222        306      0.905      0.879      0.931      0.691
                  Tram         76         99      0.919      0.929      0.947      0.812
Speed: 0.1ms preprocess, 0.7ms inference, 0.0ms loss, 0.6ms postprocess per image
Saving /workspace/yolov8nbase_noeiouloss_KD_hyperpat20_distill/eval_metrics/predictions.json...
Results saved to /workspace/yolov8nbase_noeiouloss_KD_hyperpat20_distill/eval_metrics
Validation time: 12.27 s

In [29]:
import numpy as np
import pandas as pd

# -----------------------------
# CONFIGURATION
# -----------------------------
class_names = ["Car", "Van", "Truck", "Pedestrian", "Person_sitting", "Cyclist", "Tram"]
num_classes = len(class_names)

# -----------------------------
# CONFUSION MATRIX (predicted rows x true cols)
# -----------------------------
conf_matrix = np.array([
    [5523, 10, 1, 0, 0, 0, 0, 322],
    [2, 533, 2, 0, 0, 0, 0, 45],
    [0, 0, 187, 0, 0, 0, 0, 8],
    [0, 0, 0, 790, 0, 5, 0, 111],
    [0, 0, 0, 0, 23, 0, 0, 10],
    [0, 0, 0, 1, 0, 278, 0, 40],
    [0, 1, 0, 0, 0, 0, 95, 6],
    [155, 19, 8, 105, 7, 23, 4, n]
])

# -----------------------------
# PER-CLASS LOG
# -----------------------------
per_class_precision = [0.961, 0.952, 0.971, 0.922, 0.754, 0.92, 0.919]  
per_class_recall =    [0.963, 0.952, 0.939, 0.837, 0.733, 0.895, 0.949]  
per_class_map50 = [0.986, 0.977, 0.978, 0.913, 0.741, 0.946, 0.959]
per_class_map5095 = [0.883, 0.868, 0.886, 0.585, 0.574, 0.755, 0.84]

# -----------------------------
# COMPUTE METRICS
# -----------------------------
metrics_data = []
total_samples = np.sum(conf_matrix)

for i, cname in enumerate(class_names):
    TP = conf_matrix[i, i]
    FP = np.sum(conf_matrix[i, :]) - TP
    FN = np.sum(conf_matrix[:, i]) - TP
    TN = total_samples - (TP + FP + FN)

    #Precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    #Recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    Precision = per_class_precision[i] 
    Recall = per_class_recall[i]        
    F1 = 2 * Precision * Recall / (Precision + Recall) if (Precision + Recall) > 0 else 0
    Accuracy = (TP + TN) / total_samples if total_samples > 0 else 0
    FPR = FP / (FP + TN) if (FP + TN) > 0 else 0
    FNR = FN / (FN + TP) if (FN + TP) > 0 else 0

    metrics_data.append({
        "Class": cname,
        "TP": int(TP),
        "FP": int(FP),
        "FN": int(FN),
        "TN": int(TN),
        "Precision": round(per_class_precision[i], 4),
        "Recall": round(per_class_recall[i], 4),
        "F1 Score": round(F1, 4),
        "Accuracy": round(Accuracy, 4),
        "FPR": round(FPR, 4),
        "FNR": round(FNR, 4),
        "mAP@0.5": per_class_map50[i],
        "mAP@0.5:0.95": per_class_map5095[i]
    })

# -----------------------------
# OVERALL ROW
# -----------------------------
TP_total = sum(row["TP"] for row in metrics_data)
FP_total = sum(row["FP"] for row in metrics_data)
FN_total = sum(row["FN"] for row in metrics_data)
TN_total = sum(row["TN"] for row in metrics_data)

overall = {
    "Class": "Overall",
    "TP": TP_total,
    "FP": FP_total,
    "FN": FN_total,
    "TN": TN_total,
    "Precision": np.mean([row["Precision"] for row in metrics_data]),
    "Recall": np.mean([row["Recall"] for row in metrics_data]),
    "F1 Score": np.mean([row["F1 Score"] for row in metrics_data]),
    "Accuracy": np.mean([row["Accuracy"] for row in metrics_data]),
    "FPR": np.mean([row["FPR"] for row in metrics_data]),
    "FNR": np.mean([row["FNR"] for row in metrics_data]),
    "mAP@0.5": np.mean([row["mAP@0.5"] for row in metrics_data]),
    "mAP@0.5:0.95": np.mean([row["mAP@0.5:0.95"] for row in metrics_data])
}

metrics_data.append(overall)

# -----------------------------
# SAVE TO CSV
# -----------------------------
df_metrics = pd.DataFrame(metrics_data)
df_metrics.to_csv("yolov8neiou_noeioulossinKD_hyperpat20_distill_metrics.csv", index=False)
print(df_metrics)


NameError: name 'n' is not defined

In [30]:
from ultralytics import YOLO

# Try loading and running inference with the trained model
model = YOLO("runs/detect/yolov8neiou_noeiouloss_KD_hyperpat20/weights/best.pt")  # path to your trained model
results = model.predict("/workspace/000007.png")  # use any sample image
results[0].show()


FileNotFoundError: [Errno 2] No such file or directory: 'runs/detect/yolov8neiou_noeiouloss_KD_hyperpat20/weights/best.pt'

In [16]:
import shutil

# Specify the folder path and the destination zip file path
folder_path = '/workspace/yolov8neiou_noeiouloss_KD_hyperpat20_distill'
zip_file_path = '/workspace/yolov8neiou_noeiouloss_KD_hyperpat20_distill.zip'

# Create a zip file from the folder
shutil.make_archive(zip_file_path.replace('.zip', ''), 'zip', folder_path)

'/workspace/yolov8neiou_noeiouloss_KD_hyperpat20_distill.zip'

In [17]:
import shutil

# Specify the folder path and the destination zip file path
folder_path = 'runs/detect/yolov8neiou_noeiouloss_KD_hyperpat20'
zip_file_path = 'runs/detect/yolov8neiou_noeiouloss_KD_hyperpat20.zip'

# Create a zip file from the folder
shutil.make_archive(zip_file_path.replace('.zip', ''), 'zip', folder_path)

'/workspace/yolo-distiller/runs/detect/yolov8neiou_noeiouloss_KD_hyperpat20.zip'